In [11]:
import numpy as np
import pandas as pd
import torch
import csv
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data.encoders import GroupNormalizer
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping

In [12]:
all_files = {}

all_files['nodes_all'] = "2025_Problem_D_Data/nodes_all.csv"
all_files['edges_all'] = "2025_Problem_D_Data/edges_all.csv"
all_files['nodes_drive'] = "2025_Problem_D_Data/nodes_drive.csv"
all_files['edges_drive'] = "2025_Problem_D_Data/edges_drive.csv"
all_files['Bus_Stops'] = "2025_Problem_D_Data/Bus_Stops.csv"
all_files['SHA'] = "2025_Problem_D_Data/MDOT_SHA_Annual_Average_Daily_Traffic_Baltimore.csv"
all_files['Cache'] = "saved_graphs/network_graph.pkl"

In [15]:
# Load and preprocess data (as provided)
data = []
labels = []

with open(all_files['SHA'], 'r') as f:
    reader = csv.DictReader(f)
    
    for row in reader:
        sample = []
        sorted_keys = []
        for key in row:
            if key.startswith("AADT"):
                if key[5].isdigit():
                    sorted_keys.append(key)
                elif key == 'AADT (Current)':
                    labels.append(float(row[key]))
        
        # Sort historical AADT keys chronologically
        sorted_keys.sort(key=lambda x: x[5:])  # Sort by year
        for key in sorted_keys:
            if row[key].strip():
                sample.append(float(row[key]))
        
        data.append(sample)

In [22]:
print(sorted_keys)

['AADT 2014', 'AADT 2015', 'AADT 2016', 'AADT 2017', 'AADT 2018', 'AADT 2019', 'AADT 2020', 'AADT 2021', 'AADT 2022']


In [14]:
import numpy as np
import csv
from keras.models import Sequential
from keras.layers import LSTM, Dense
from keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Load and preprocess data (as provided)
data = []
labels = []

with open(all_files['SHA'], 'r') as f:
    reader = csv.DictReader(f)
    
    for row in reader:
        sample = []
        sorted_keys = []
        for key in row:
            if key.startswith("AADT"):
                if key[5].isdigit():
                    sorted_keys.append(key)
                elif key == 'AADT (Current)':
                    labels.append(float(row[key]))
        
        # Sort historical AADT keys chronologically
        sorted_keys.sort(key=lambda x: x[5:])  # Sort by year
        for key in sorted_keys:
            if row[key].strip():
                sample.append(float(row[key]))
        
        data.append(sample)

# Convert to numpy arrays and pad sequences
max_length = max(len(seq) for seq in data)
X = pad_sequences(data, maxlen=max_length, padding='post', dtype='float32')
X = X.reshape((X.shape[0], X.shape[1], 1))  # Reshape for LSTM input
y = np.array(labels)

# Split data into train/test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build LSTM model
model = Sequential()
model.add(LSTM(64, activation='relu', input_shape=(max_length, 1)))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train the model
history = model.fit(X_train, y_train, 
                    epochs=100, 
                    batch_size=32, 
                    validation_data=(X_test, y_test),
                    verbose=1)

# Evaluate model
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test MAE: {test_mae:.4f}")

# Example prediction (using first sample)
sample_input = X[0].reshape(1, max_length, 1)
predicted_aadt = model.predict(sample_input)
print(f"Predicted AADT: {predicted_aadt[0][0]:.1f}")
print(f"Actual AADT: {y[0]}")

Epoch 1/100


c:\Users\USER\anaconda3\envs\forecasting_env\lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 210745472.0000 - mae: 6027.0054 - val_loss: 10383007.0000 - val_mae: 1363.3051
Epoch 2/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 9524828.0000 - mae: 1228.2489 - val_loss: 6754630.0000 - val_mae: 1130.8391
Epoch 3/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 8230356.5000 - mae: 1174.4237 - val_loss: 5548378.5000 - val_mae: 1095.6976
Epoch 4/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 7282514.0000 - mae: 1140.1510 - val_loss: 12578079.0000 - val_mae: 1482.0138
Epoch 5/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 8591026.0000 - mae: 1251.8801 - val_loss: 7085587.5000 - val_mae: 1071.9330
Epoch 6/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 6349383.5000 - mae: 1046.9460 - val_loss: 6422245.5000 - val_mae: 1057.7229
Epoch 7/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 8789633.0000 - mae: 1200.2762 - val_loss: 54621692.0000 - val_mae: 3193.6926
Epoch 8/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 155

In [21]:
# Add this after training
# After calculating max_length
np.save('models/max_length.npy', max_length)
model.save('models/aadt_predictor.keras') 